In [ ]:
!pip install optuna lightgbm pandas scikit-learn matplotlib

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load data (replace with your dataset path)
X_train = pd.read_parquet('temp/X_resampled.parquet')
X_val = pd.read_parquet('temp/X_val.parquet')


y_train = X_train['TARGET']
y_val = X_val['TARGET']

X_train = X_train.drop(columns=['TARGET'])
X_val = X_val.drop(columns=['TARGET'])

In [2]:
import pandas as pd
df = pd.read_csv('temp/feature_importance.csv')
df.to_excel('temp/feature_importance.xlsx', index=False)

In [ ]:
X_train

In [ ]:
X_train.columns = X_train.columns.str.replace(r'[^\w]', '_', regex=True)
X_val.columns = X_val.columns.str.replace(r'[^\w]', '_', regex=True)

In [ ]:
import optuna
import lightgbm as lgb
import numpy as np

def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-3, 10.0),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-3, 10.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        'n_jobs': -1
    }
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        param,
        train_data,
        valid_sets=[valid_data],
    )
    
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    return auc


In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=500)

# Print the best parameters
print("Best parameters:", study.best_params)
print("Best AUC:", study.best_value)

In [ ]:
best_params = study.best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

In [ ]:
param = {'learning_rate': 0.07489690004483521, 'num_leaves': 102, 'max_depth': 15, 'min_data_in_leaf': 189, 'feature_fraction': 0.5372551032070825, 'bagging_fraction': 0.810500564688185, 'bagging_freq': 1, 'lambda_l1': 6.0455562622501855, 'lambda_l2': 0.293317330622806, 'min_child_samples': 82}



In [ ]:
best_params

In [ ]:
best_params = param
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

print("Training the final model with the best parameters")
print(best_params)

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=1000
)

In [ ]:
y_pred = final_model.predict(X_val)

In [ ]:
roc_auc_score(y_val,y_pred)

In [ ]:
# Get feature importance and feature names
importance = final_model.feature_importance(importance_type='gain')  # 'gain' measures the contribution
feature_names = X_train.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

# Display the most important feature
best_feature = importance_df.iloc[0]
print(f"The most important feature is: {best_feature['Feature']} with an importance score of {best_feature['Importance']}")

# Optional: Display the top 5 features
print("\nTop 5 Features:")
print(importance_df.head())


In [ ]:
importance_df.to_csv('temp/feature_importance.csv', index=False)